## Imports

In [1]:
import os

from alphagenome.data import genome
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models import dna_client
import matplotlib.pyplot as plt
import pandas as pd

## Local Model Alpha Genome

In [2]:
# Load model using persistent HF auth and local FASTA override
import os
from pathlib import Path
from getpass import getpass
from huggingface_hub import whoami, login
from alphagenome_research.model import dna_model
from alphagenome.models import dna_model as public_dna_model


def ensure_hf_auth(allow_prompt=True):
    try:
        info = whoami()
        print(f"HF already authenticated as: {info.get('name', 'unknown')}")
        return
    except Exception:
        pass

    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False, skip_if_logged_in=True)
        info = whoami()
        print(f"HF authenticated via environment token as: {info.get('name', 'unknown')}")
        return

    if allow_prompt:
        try:
            token = getpass("Enter Hugging Face token (input hidden): ").strip()
        except Exception:
            token = ""

        if token:
            login(token=token, add_to_git_credential=False, skip_if_logged_in=True)
            info = whoami()
            print(f"HF authenticated via interactive prompt as: {info.get('name', 'unknown')}")
            return

    raise RuntimeError(
        "No Hugging Face credentials found. Use one of:\n"
        "1) Run `hf auth login` in a terminal\n"
        "2) Set HF_TOKEN (or HUGGINGFACEHUB_API_TOKEN)\n"
        "3) Re-run this cell and paste a token when prompted"
    )


def resolve_local_human_fasta_path():
    candidates = [
        os.getenv("ALPHAGENOME_FASTA"),
        "/data/Dcode/gaetano/repos/fasta_files/hg38.fa",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


def resolve_local_mouse_fasta_path():
    candidates = [
        os.getenv("ALPHAGENOME_MOUSE_FASTA"),
        "/data/Dcode/gaetano/repos/fasta_files/mm10.fa",
        "/data/Dcode/gaetano/repos/fasta_files/mm39.fa",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


ensure_hf_auth(allow_prompt=True)

human_fasta = resolve_local_human_fasta_path()
if not human_fasta:
    raise FileNotFoundError(
        "No local human FASTA found. Set ALPHAGENOME_FASTA to a local GRCh38 FASTA path, e.g.\n"
        "export ALPHAGENOME_FASTA=/path/to/GRCh38.p13.genome.fa"
    )

mouse_fasta = resolve_local_mouse_fasta_path()
if mouse_fasta:
    print(f"Using local mouse FASTA: {mouse_fasta}")
else:
    print(
        "No local mouse FASTA found; configuring mouse organism without local FASTA. "
        "Predictions on human intervals still work."
    )

print(f"Using local human FASTA: {human_fasta}")
organism_settings = {
    public_dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings(
        fasta_path=human_fasta,
    ),
    public_dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings(
        fasta_path=mouse_fasta,
    ) if mouse_fasta else dna_model.OrganismSettings(),
}

model = dna_model.create_from_huggingface("fold-1", organism_settings=organism_settings)
print("Model loaded successfully.")

HF already authenticated as: tanoManzo
No local mouse FASTA found; configuring mouse organism without local FASTA. Predictions on human intervals still work.
Using local human FASTA: /data/Dcode/gaetano/repos/fasta_files/hg38.fa


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Model loaded successfully.


In [ ]:
dna_client.OutputType.

<enum 'OutputType'>

## Load Data (Enhancers)

### HepG2 Enhancer Training Data

In [4]:
# HepG2 Enhancer
path_training_data = "/vf/users/Dcode/gaetano/FootPrinting/data/input_training_trednet/enhancers_dhs_control"
path_positive_data = f"{path_training_data}/HepG2_Enhancer_DHS_positive_1kb.bed"
path_control_data = f"{path_training_data}/HepG2_Enhancer_DHS_control_1kb.bed"

bed_cols = ["chrom", "start", "end"]
positive_hepg2_enh = pd.read_csv(path_positive_data, sep="\t", header=None, names=bed_cols)
control_hepg2_enh = pd.read_csv(path_control_data, sep="\t", header=None, names=bed_cols)


positive_hepg2_enh.head(), control_hepg2_enh.head()

(  chrom    start      end
 0  chr1   190392   191392
 1  chr1   726724   727724
 2  chr1  1006170  1007170
 3  chr1  1017069  1018069
 4  chr1  1079090  1080090,
   chrom   start     end
 0  chr1   19290   20290
 1  chr1  108689  109689
 2  chr1  189280  190280
 3  chr1  592693  593693
 4  chr1  603750  604750)

### Convert into Alpha Genome Intervals

In [5]:
sample_size = 20000

positive_intervals = []
for index, row in positive_hepg2_enh.sample(sample_size).iterrows():
    positive_intervals.append(genome.Interval(chromosome=row['chrom'], start=row['start'], end=row['end']))

control_intervals = []
for index, row in control_hepg2_enh.sample(sample_size).iterrows():
    control_intervals.append(genome.Interval(chromosome=row['chrom'], start=row['start'], end=row['end']))

## Predictions 

In [10]:
16384/128

128.0

In [5]:
import gc
import h5py
import numpy as np
from tqdm.auto import tqdm

MODALITIES = {
    "atac":         (16384,  167),
    "dnase":        (16384,  305),
    "procap":       (16384,   12),
    "chip_histone": (  128, 1116),
}

def predict_chunk(intervals, sequence_length):
    """Run inference on a list of intervals, return list of Output objects."""
    outputs = []
    for iv in tqdm(intervals, desc="  inference", leave=False):
        try:
            iv = iv.resize(sequence_length)
            out = model.predict_interval(
                interval=iv,
                requested_outputs=list(dna_model.OutputType),
                ontology_terms=None,
            )
            outputs.append(out)
        except Exception as e:
            print(f"  Error: {e} — skipping")
            outputs.append(None)
    return outputs


def append_chunk_to_hdf5(outputs, f, grp, start_idx):
    """Write a chunk of Output objects into pre-allocated HDF5 datasets."""
    for j, out in enumerate(outputs):
        if out is None:
            continue
        i = start_idx + j
        grp["atac"][i]         = out.atac.values
        grp["dnase"][i]        = out.dnase.values
        grp["procap"][i]       = out.procap.values
        grp["chip_histone"][i] = out.chip_histone.values
    grp.attrs["last_written_idx"] = start_idx + len(outputs) - 1


def predict_and_save_chunked(
    intervals,
    label,
    out_path,
    chunk_size=200,
    sequence_length=None,
):
    """
    Process intervals in chunks of `chunk_size`.
    Each chunk is inferred, written to HDF5, then freed from memory.
    Supports resuming from last written index on crash.
    """
    if sequence_length is None:
        sequence_length = dna_client.SEQUENCE_LENGTH_16KB

    n = len(intervals)

    with h5py.File(out_path, "a") as f:
        grp = f.require_group(label)

        # Pre-allocate datasets once
        for mod, (pos, ch) in MODALITIES.items():
            if mod not in grp:
                grp.create_dataset(
                    mod,
                    shape=(n, pos, ch),
                    dtype="float32",
                    chunks=(min(chunk_size, 32), pos, ch),  # HDF5 chunk ≤ inference chunk
                )

        # Resume support
        last_written = grp.attrs.get("last_written_idx", -1)
        start = last_written + 1
        if start > 0:
            print(f"Resuming {label} from index {start}/{n}")

        # Process in chunks
        for chunk_start in tqdm(range(start, n, chunk_size),
                                desc=f"Chunks [{label}]"):
            chunk_end = min(chunk_start + chunk_size, n)
            chunk_intervals = intervals[chunk_start:chunk_end]

            outputs = predict_chunk(chunk_intervals, sequence_length)
            append_chunk_to_hdf5(outputs, f, grp, chunk_start)
            f.flush()  # ensure data hits disk

            # Free memory
            del outputs
            gc.collect()

    print(f"✓ Done [{label}]: {n} sequences saved to {out_path}")


# ── Entry point ────────────────────────────────────────────

HDF5_PATH = "enhancer_features.h5"

predict_and_save_chunked(positive_intervals, "positive", HDF5_PATH, chunk_size=200)
predict_and_save_chunked(control_intervals,  "control",  HDF5_PATH, chunk_size=200)

# Free model from memory
del model
gc.collect()
try:
    import jax
    jax.clear_caches()
except Exception:
    pass
print("✓ Model unloaded")

Chunks [positive]:   0%|          | 0/100 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

... (194 outputs left out)

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

  inference:   0%|          | 0/200 [00:00<?, ?it/s]

✓ Done [control]: 20000 sequences saved to enhancer_features.h5
✓ Model unloaded


#### Output Example

In [ ]:
# 2) Iterate through every output type and print what's present
outputs = all_outputs_positive[0]
for ot in dna_model.OutputType:
    item = outputs.get(ot)
    if item is not None:
        vals = getattr(item, "values", None)
        shape = getattr(vals, "shape", None)
        print(f"{ot.name}: {type(item).__name__}, shape={shape}")

## CNN

In [1]:
import json
import numpy as np
import h5py
import keras
from keras import layers, Model

MODALITIES = {
    "atac":         (16384,  167),
    "dnase":        (16384,  305),
    "procap":       (16384,   12),
    "chip_histone": (  128, 1116),
}

# ─────────────────────────────────────────────
# 1. HDF5 Data Generator
# ─────────────────────────────────────────────

class HDF5EnhancerDataset(keras.utils.PyDataset):
    """
    Reads batches directly from HDF5 — never loads full dataset into RAM.
    indices: array of (label, row_idx) tuples, e.g. [("positive", 0), ("control", 5), ...]
    """
    def __init__(self, h5_path, indices, batch_size=32, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.h5_path    = h5_path
        self.indices    = np.array(indices, dtype=object)
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self._order     = np.arange(len(self.indices))
        if self.shuffle:
            np.random.shuffle(self._order)

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, batch_idx):
        batch_order = self._order[
            batch_idx * self.batch_size : (batch_idx + 1) * self.batch_size
        ]
        batch_indices = self.indices[batch_order]

        X = {mod: [] for mod in MODALITIES}
        y = []

        with h5py.File(self.h5_path, "r") as f:
            for label, row in batch_indices:
                row = int(row)
                for mod in MODALITIES:
                    X[mod].append(f[label][mod][row])
                y.append(1.0 if label == "positive" else 0.0)

        return (
            {mod: np.stack(X[mod]).astype(np.float32) for mod in MODALITIES},
            np.array(y, dtype=np.float32),
        )

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self._order)


def build_index(h5_path):
    """
    Build a flat list of (label, row_idx) tuples from the HDF5 file.
    Also returns class counts for computing class weights.
    """
    indices = []
    counts  = {}
    with h5py.File(h5_path, "r") as f:
        for label in ["positive", "control"]:
            if label not in f:
                print(f"Warning: '{label}' group not found in {h5_path}")
                continue
            n = f[label]["atac"].shape[0]
            counts[label] = n
            for i in range(n):
                indices.append((label, i))
    return indices, counts


def train_val_split(indices, counts, val_frac=0.1, seed=42):
    """Stratified split preserving positive/control ratio."""
    rng = np.random.default_rng(seed)
    pos_idx  = [(l, i) for l, i in indices if l == "positive"]
    ctrl_idx = [(l, i) for l, i in indices if l == "control"]

    def split(arr):
        arr = list(arr)
        rng.shuffle(arr)
        n_val = max(1, int(len(arr) * val_frac))
        return arr[n_val:], arr[:n_val]

    pos_tr,  pos_val  = split(pos_idx)
    ctrl_tr, ctrl_val = split(ctrl_idx)

    return pos_tr + ctrl_tr, pos_val + ctrl_val


# ─────────────────────────────────────────────
# 2. Model (unchanged from before)
# ─────────────────────────────────────────────

def modality_encoder(inp, out_channels=64, name="mod"):
    input_len = inp.shape[1]
    x = layers.Conv1D(out_channels, kernel_size=1, padding="same",
                      name=f"{name}_proj")(inp)
    x = layers.BatchNormalization(name=f"{name}_bn0")(x)
    x = layers.ReLU(name=f"{name}_relu0")(x)
    if input_len == 16384:
        for i, (ks, st) in enumerate([(8,4), (8,4), (4,4), (2,2)]):
            x = layers.AveragePooling1D(pool_size=ks, strides=st, padding="same",
                                        name=f"{name}_pool{i}")(x)
    return x


def shared_cnn(x):
    for filters, ks, pool in [(256,9,2), (128,9,2), (128,5,2), (64,5,None)]:
        x = layers.Conv1D(filters, kernel_size=ks, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        if pool:
            x = layers.MaxPooling1D(pool_size=pool)(x)
    return layers.GlobalAveragePooling1D()(x)


def build_model():
    inp_atac  = keras.Input(shape=(16384,  167), name="atac")
    inp_dnase = keras.Input(shape=(16384,  305), name="dnase")
    inp_procap= keras.Input(shape=(16384,   12), name="procap")
    inp_chiph = keras.Input(shape=(  128, 1116), name="chip_histone")

    fused = layers.Concatenate(axis=-1, name="fusion")([
        modality_encoder(inp_atac,   name="atac"),
        modality_encoder(inp_dnase,  name="dnase"),
        modality_encoder(inp_procap, name="procap"),
        modality_encoder(inp_chiph,  name="chiph"),
    ])

    emb = shared_cnn(fused)
    x   = layers.Dense(64, activation="relu",
                        kernel_regularizer=keras.regularizers.L2(1e-4))(emb)
    x   = layers.Dropout(0.4)(x)  # increased from 0.3
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return Model(inputs=[inp_atac, inp_dnase, inp_procap, inp_chiph],
                 outputs=out, name="enhancer_cnn")


# ─────────────────────────────────────────────
# 3. Training
# ─────────────────────────────────────────────

def train(h5_path, batch_size=64):  # increased from 32
    # Build index and split
    indices, counts = build_index(h5_path)
    print(f"Dataset: {counts}")

    tr_idx, val_idx = train_val_split(indices, counts, val_frac=0.1)
    print(f"Train: {len(tr_idx)} | Val: {len(val_idx)}")

    # Class weights
    n_pos  = counts.get("positive", 1)
    n_ctrl = counts.get("control",  1)
    class_weight = {0: 1.0, 1: n_ctrl / n_pos}
    print(f"Class weights: {class_weight}")

    # Generators
    train_ds = HDF5EnhancerDataset(h5_path, tr_idx,  batch_size=batch_size, shuffle=True)
    val_ds   = HDF5EnhancerDataset(h5_path, val_idx, batch_size=batch_size, shuffle=False)

    # Build and compile
    model = build_model()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(name="auroc", curve="ROC"),
            keras.metrics.AUC(name="auprc", curve="PR"),
            keras.metrics.BinaryAccuracy(name="acc"),
        ]
    )
    model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_auroc", mode="max",
            patience=8,                      # increased from 5
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auroc", mode="max",
            factor=0.5, patience=5,          # increased from 2
            min_lr=1e-6
        ),
        keras.callbacks.ModelCheckpoint(
            "best_enhancer_cnn.keras",
            monitor="val_auroc", mode="max",
            save_best_only=True
        ),
        keras.callbacks.CSVLogger("training_log.csv"),  # save metrics to disk
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    # Save history for later plotting
    with open("training_history.json", "w") as f:
        json.dump(history.history, f)
    print("Saved training history → training_history.json")

    return model, history


# ─────────────────────────────────────────────
# 4. Entry point
# ─────────────────────────────────────────────

if __name__ == "__main__":
    model, history = train("enhancer_features.h5", batch_size=64)

2026-05-20 09:26:13.204019: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Dataset: {'positive': 20000, 'control': 20000}
Train: 36000 | Val: 4000
Class weights: {0: 1.0, 1: 1.0}


I0000 00:00:1779283578.437555  218015 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79189 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:46:00.0, compute capability: 8.0


Model: "enhancer_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ atac (InputLayer)   │ (None, 16384,     │          0 │ -                 │
│                     │ 167)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnase (InputLayer)  │ (None, 16384,     │          0 │ -                 │
│                     │ 305)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ procap (InputLayer) │ (None, 16384, 12) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atac_proj (Conv1D)  │ (None, 16384, 64) │     10,752 │ atac[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnase_proj (Conv1D) │ (None, 16384, 64) │     19,584 │ dnase[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ procap_proj         │ (None, 16384, 64) │        832 │ procap[0][0]      │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atac_bn0            │ (None, 16384, 64) │        256 │ atac_proj[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnase_bn0           │ (None, 16384, 64) │        256 │ dnase_proj[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ procap_bn0          │ (None, 16384, 64) │        256 │ procap_proj[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atac_relu0 (ReLU)   │ (None, 16384, 64) │          0 │ atac_bn0[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnase_relu0 (ReLU)  │ (None, 16384, 64) │          0 │ dnase_bn0[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ procap_relu0 (ReLU) │ (None, 16384, 64) │          0 │ procap_bn0[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atac_pool0          │ (None, 4096, 64)  │          0 │ atac_relu0[0][0]  │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnase_pool0         │ (None, 4096, 64)  │          0 │ dnase_relu0[0][0] │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ procap_pool0        │ (None, 4096, 64)  │          0 │ procap_relu0[0][… │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chip_histone        │ (None, 128, 1116) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atac_pool1          │ (None, 1024, 64)  │          0 │ atac_pool0[0][0]  │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnase_pool1         │ (None, 1024, 64)  │          0 │ dnase_pool0[0][0] │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ procap_pool1        │ (None, 1024, 64)  │          0 │ procap_pool0[0][

 Total params: 1,118,401 (4.27 MB)

... (3 outputs left out)

2026-05-20 09:26:39.303185: I external/local_xla/xla/service/service.cc:163] XLA service 0x1553c80036f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-20 09:26:39.303204: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2026-05-20 09:26:39.397074: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-20 09:26:40.008409: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-05-20 09:26:40.284962: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-20 09:26:40.284987: I ext

  1/563 ━━━━━━━━━━━━━━━━━━━━ 3:44:39 24s/step - acc: 0.6250 - auprc: 0.6163 - auroc: 0.6768 - loss: 0.6896

I0000 00:00:1779283614.116911  219889 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


563/563 ━━━━━━━━━━━━━━━━━━━━ 3666s 6s/step - acc: 0.8484 - auprc: 0.9071 - auroc: 0.9162 - loss: 0.3745 - val_acc: 0.8685 - val_auprc: 0.9456 - val_auroc: 0.9506 - val_loss: 0.3461 - learning_rate: 0.0010
Epoch 2/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 3831s 7s/step - acc: 0.9153 - auprc: 0.9672 - auroc: 0.9665 - loss: 0.2334 - val_acc: 0.9107 - val_auprc: 0.9776 - val_auroc: 0.9781 - val_loss: 0.2631 - learning_rate: 0.0010
Epoch 3/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 3822s 7s/step - acc: 0.9373 - auprc: 0.9789 - auroc: 0.9783 - loss: 0.1841 - val_acc: 0.9335 - val_auprc: 0.9837 - val_auroc: 0.9832 - val_loss: 0.1826 - learning_rate: 0.0010
Epoch 4/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 3812s 7s/step - acc: 0.9467 - auprc: 0.9838 - auroc: 0.9837 - loss: 0.1593 - val_acc: 0.9087 - val_auprc: 0.9836 - val_auroc: 0.9852 - val_loss: 0.2611 - learning_rate: 0.0010
Epoch 5/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 3789s 7s/step - acc: 0.9503 - auprc: 0.9856 - auroc: 0.9854 - loss: 0.1490 - val_acc: 0.8900 - val_auprc: 0